In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df=pd.read_csv('titanic.csv')

print(f"Dataframe shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nDataframe First 5 Records:")
display(df.head(5))

print("\nData types and non-null counts:")
print(df.info())

print("\nmissing values:")
print(df.isnull().sum())

print("\nTarget Class Balance:")
print(df['Survived'].value_counts(normalize=True).round(4)*100)

Dataframe shape: 891 rows x 12 columns

Dataframe First 5 Records:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB
None

missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin        

In [4]:
print(f"Survival Rate by Gender:")
print(f"{df.groupby('Sex')['Survived'].mean().round(4)*100}")

print(f"\nSurvival Rate by Passenger Class:")
print(f"{df.groupby('Pclass')['Survived'].mean().round(4)*100}")

print(f"\nSurvival Rate by Embarked:")
print(f"{df.groupby('Embarked')['Survived'].mean().round(4)*100}")

Survival Rate by Gender:
Sex
female    74.20
male      18.89
Name: Survived, dtype: float64

Survival Rate by Passenger Class:
Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64

Survival Rate by Embarked:
Embarked
C    55.36
Q    38.96
S    33.70
Name: Survived, dtype: float64


In [5]:

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1, inplace=True)
print(df.isnull().sum())
display(df.head())

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

x = df.drop("Survived", axis=1)
y = df["Survived"]

categorical_cols =["Sex", "Embarked","Pclass"]
numerical_cols = ["Age", "SibSp", "Parch", "Fare"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(), categorical_cols)
    ]
)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {x_train.shape[0]} rows x {x_train.shape[1]} columns")

Training set shape: 712 rows x 7 columns


In [7]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Patient_Readmission_Prediction")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []

for name, clf in models.items():
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
        pipeline.fit(x_train, y_train)
       
        y_pred = pipeline.predict(x_test)
        y_prob = pipeline.predict_proba(x_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)

        mlflow.log_param("model", name)
        mlflow.log_params(clf.get_params())
        mlflow.log_metrics({"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1, "roc_auc": roc_auc})    
        mlflow.sklearn.log_model(pipeline, artifact_path="model", serialization_format="cloudpickle")

        print(f"{name}: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, ROC_AUC={roc_auc:.4f}")
        results.append({"name": name, "f1": f1, "pipeline": pipeline})


2026/10/05 21:33:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/05 21:33:00 INFO mlflow.store.db.utils: Updating database tables
2026/10/05 21:33:01 INFO mlflow.tracking.fluent: Experiment with name 'Patient_Readmission_Prediction' does not exist. Creating a new experiment.
2026/10/05 21:33:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/05 21:33:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression: Accuracy=0.8045, Precision=0.7931, Recall=0.6667, F1=0.7244, ROC_AUC=0.8432


2026/10/05 21:33:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/05 21:33:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/05 21:33:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest: Accuracy=0.7989, Precision=0.7895, Recall=0.6522, F1=0.7143, ROC_AUC=0.8415


2026/10/05 21:33:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gradient Boosting: Accuracy=0.7933, Precision=0.7857, Recall=0.6377, F1=0.7040, ROC_AUC=0.8189


In [10]:
import joblib
results.sort(key=lambda x: x["f1"], reverse=True)
best_model = results[0]["pipeline"]
print(f"Best model: {results[0]['name']} with F1 score: {results[0]['f1']:.4f}")

joblib.dump(best_model, "best_model.pkl")
print("Best model saved as 'best_model.pkl'")

best_run = mlflow.search_runs(
    experiment_names=["Patient_Readmission_Prediction"],
    order_by=["metrics.f1_score DESC"],
).iloc[0]

model_uri = f"runs:/{best_run.run_id}/model"
registered_model = mlflow.register_model(model_uri, "Patient_Readmission_Prediction")
print(f"Best model registered as 'Patient_Readmission_Prediction' with run ID: {best_run.run_id}")

Best model: Logistic Regression with F1 score: 0.7244
Best model saved as 'best_model.pkl'


Successfully registered model 'Patient_Readmission_Prediction'.
2026/10/05 21:40:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 2d9537892550492791fd15610e41bd11 has no artifacts at artifact path 'model', registering model based on models:/m-eba5dad3e18345acaa02bc5d4fd5f97d instead


Best model registered as 'Patient_Readmission_Prediction' with run ID: 2d9537892550492791fd15610e41bd11


Created version '1' of model 'Patient_Readmission_Prediction'.


In [13]:
import joblib
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pandas as pd


model = joblib.load("best_model.pkl")

class Patientinput(BaseModel):
    Pclass: int = Field(..., example=1)
    Sex: str = Field(..., example="female")
    Age: float = Field(..., example=38.0)
    SibSp: int = Field(..., example=1)
    Parch: int = Field(..., example=0)
    Fare: float = Field(..., example=71.28)
    Embarked: str = Field(..., example="C")

app = FastAPI(title="Patient Readmission Prediction", version="1.0")

@app.get("/health")
def health():
    return {"status": "API is running", "model_loaded": True}

@app.post("/predict")
def predict_readmission(patient: Patientinput):
    df_patient = pd.DataFrame([patient.model_dump()])
    probability = float(model.predict_proba(df_patient)[:, 1][0])
    prediction = int(model.predict(df_patient)[0])

    return {"readmission_probability":round(probability,4),
            "prediction": prediction,
            "risk_level": "High Risk" if prediction == 1 else "Low Risk",
    }

C:\Users\saifp\AppData\Local\Temp\ipykernel_15004\3670673725.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  Pclass: int = Field(..., example=1)
C:\Users\saifp\AppData\Local\Temp\ipykernel_15004\3670673725.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  Sex: str = Field(..., example="female")
C:\Users\saifp\AppData\Local\Temp\ipykernel_15004\3670673725.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead

In [22]:
import pandas as pd
import numpy as np
from evidently import Report
from evidently.presets import DataDriftPreset


if not hasattr(np, 'float_'):
    np.float_= np.float64

reference = x_train
current = x_test

report = Report(metrics=[DataDriftPreset()])

results = report.run(reference_data=reference, current_data=current)

html = results._repr_html_()
with open("evidently_report.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Report Successfully Generated: 'evidently_report.html'")

Report Successfully Generated: 'evidently_report.html'
